# 1kg_eur — rare-variant PCA with relateds removed

Diagnostic for the strange loadings in the rare-variant PCA. Relatedness is the
standard explanation: a family cluster shares long haplotypes, so its members
carry the *same rare alleles at the same loci*. For a rare variant, eight
relatives carrying it can be most of that variant's variance in the whole
sample — so the family becomes a large genuinely-correlated block and PCA spends
a component on it, producing a PC that a handful of people load extremely on,
driven by variants in a few regions. Common variants are far less exposed.

The paper we are following (`s41467-026-73776-9`) excludes everyone related to
**third degree or closer** before fitting rare PCs. Our round-2 keep list does
not, because the rest of this pipeline *wants* relatives — they are the signal.

**Reuses the existing run, no rebuild.** The panel and pruned variant lists from
`01c_rare_variant_pca` are already on this VM at `~/scratch_1kg_eur_rare/`:

| file | what it is |
|---|---|
| `qc/rare_qc.{pgen,pvar,psam}` | 0.1% ≤ MAF < 1%, `--geno 0.05`, biallelic, `--rm-dup exclude-all` |
| `hm3_rare_all.prune.in` | HM3-agreeing arm, pruned `1000kb 1 0.1`, long-range LD regions excluded |
| `nonhm3_rare_all.prune.in` | non-HM3 arm, same pruning |

So the **variant set is held fixed at exactly what produced the strange loadings**
and the only thing that changes is which individuals enter the eigendecomposition.
Nothing is re-derived, nothing is downloaded, no Batch job — everything here runs
locally on the VM in minutes, the same way `01c`'s own pruning and PCA cells did.

Note this also rules out one suspect up front: `01c` already passed
`--exclude bed1 high_ld_regions_grch38.txt`, so the long-range LD regions and the
MHC were **already excluded** when the odd loadings appeared. They are not the cause.

**Unrelated set** comes from the AoU aux relatedness table (`kin > 0.0442`),
greedily pruned by repeatedly dropping the individual with the most remaining
relationships. Alternatives rejected: `06a_close_pair_screen` covers only
a_ij > 0.2 and lists both members of each pair; nb07's KING output is 1st-degree
only; the 0.1% GRM shards are on the 50K subsample, not this sample;
`plink2 --king-cutoff` is the O(N²) problem nb07 hit at 227K.

## Config

In [ ]:
import os, subprocess
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SAMPLE_SET = "1kg_eur"
WS_GS = "gs://cloned-shared-env-pilot-wb-swift-sprout-7231/phenotypic_covariance_v9"
R_GS  = f"{WS_GS}/{SAMPLE_SET}"

# ── the existing 01c run on this VM ───────────────────────────────────────────
RARE   = os.path.expanduser(f"~/scratch_{SAMPLE_SET}_rare")
PANEL  = f"{RARE}/qc/rare_qc"                 # pgen/pvar/psam
ARMS   = ["hm3_rare_all", "nonhm3_rare_all"]  # each has a {arm}.prune.in
KEEP_ALL = f"{RARE}/round2_keep.txt"          # written below if absent

# ── AoU aux relatedness (same file 01_round1_gate.ipynb reads) ────────────────
AUX      = os.path.expanduser(
    "~/workspace/cdrv9/vwb-aou-datasets-controlled-v9/v9/wgs/short_read/snpindel/aux")
REL_PATH = f"{AUX}/relatedness/samples_relatedness.tsv"
REL_COLS = ("i.s", "j.s", "kin")

KEEP_GS = f"{R_GS}/01_ancestry/round2/1kg_eur_keep_ids.txt"   # pre-rename filename
OUT_GS  = f"{R_GS}/01_ancestry/rare_pca_unrelated"

# ── outputs (local) ───────────────────────────────────────────────────────────
OUT = f"{RARE}/unrel"
os.makedirs(OUT, exist_ok=True)
UNREL_KEEP = f"{OUT}/unrelated_keep_ids.txt"

# ── parameters ────────────────────────────────────────────────────────────────
# KING kinship: 1st degree ~0.25, 2nd ~0.125, 3rd ~0.0625. 0.0442 is the
# conventional 3rd/4th-degree midpoint - "3rd degree or closer", matching the
# paper and plink2's --king-cutoff default.
KIN_CUT   = 0.0442
N_PCS_FIT = 10     # matches 01c, so the 'all' fit reproduces the original run

print(f"panel: {PANEL}")
print(f"out:   {OUT}")
for f in (f"{PANEL}.pgen", f"{PANEL}.pvar", f"{PANEL}.psam"):
    print(f"  {'ok  ' if os.path.isfile(f) else 'MISS'} {f}")
for a in ARMS:
    p = f"{RARE}/{a}.prune.in"
    n = sum(1 for _ in open(p)) if os.path.isfile(p) else None
    print(f"  {'ok  ' if n else 'MISS'} {p}" + (f"  ({n:,} variants)" if n else ""))

## Check plink2 and the panel sample list

`--pca approx` on this panel at ~222K samples is what `01c` already did locally,
so the VM can take it. The `.psam` sample count is the reference for everything
below: the unrelated keep list is intersected against it, not against the bucket
keep list, so a sample that was dropped by QC cannot silently reappear.

In [ ]:
subprocess.run(["bash", "-c", f"""
export PATH="$HOME/bin:$PATH"
plink2 --version
echo "panel variants: $(grep -vc '^##' "{PANEL}.pvar")"
echo "panel samples:  $(($(wc -l < "{PANEL}.psam") - 1))"
"""], check=True)

# sample IDs actually in the panel
_psam = pd.read_csv(f"{PANEL}.psam", sep=r"\s+")
_idc  = "#IID" if "#IID" in _psam.columns else "IID"
panel_ids = set(_psam[_idc].astype(str))
print(f"\npanel sample ids: {len(panel_ids):,}")

## Build the unrelated keep list

Greedy max-independent-set: while any relationship remains, drop the individual
involved in the most of them. Not optimal (exact is NP-hard) but it is what
plink's `--king-cutoff` does, it is deterministic here, and it keeps far more
people than dropping both members of every pair — on a hub of one person related
to four others, dropping the hub saves all four.

In [ ]:
assert os.path.isfile(REL_PATH), f"{REL_PATH} missing - check {os.path.dirname(REL_PATH)}"
i_col, j_col, k_col = REL_COLS

rel_all = pd.read_csv(REL_PATH, sep="\t", usecols=list(REL_COLS),
                      dtype={i_col: str, j_col: str})
print(f"aux relatedness rows: {len(rel_all):,}")

# restrict to pairs with BOTH members in the panel
rel_all = rel_all[rel_all[i_col].isin(panel_ids) & rel_all[j_col].isin(panel_ids)]
rel = rel_all[rel_all[k_col] > KIN_CUT]
print(f"within-panel pairs at kin > {KIN_CUT}: {len(rel):,}")
print(f"individuals involved: {len(set(rel[i_col]) | set(rel[j_col])):,}")

In [ ]:
adj = defaultdict(set)
for a, b in zip(rel[i_col], rel[j_col]):
    if a != b:
        adj[a].add(b)
        adj[b].add(a)

dropped = set()
while True:
    active = [(len(v), k) for k, v in adj.items() if v]
    if not active:
        break
    _deg, worst = max(active, key=lambda t: (t[0], t[1]))   # tie-break on id
    dropped.add(worst)
    for nb in adj[worst]:
        adj[nb].discard(worst)
    adj[worst] = set()

unrelated = sorted(panel_ids - dropped)
print(f"dropped:   {len(dropped):,}")
print(f"unrelated: {len(unrelated):,} of {len(panel_ids):,} "
      f"({100 * len(unrelated) / len(panel_ids):.1f}%)")

_surv = rel[~rel[i_col].isin(dropped) & ~rel[j_col].isin(dropped)]
assert len(_surv) == 0, f"{len(_surv)} related pairs survived the prune"
print("check: no related pair survives")

In [ ]:
# plink2 accepts a single-column IID list with a #IID header
with open(UNREL_KEEP, "w") as fh:
    fh.write("#IID\n" + "\n".join(unrelated) + "\n")
with open(KEEP_ALL, "w") as fh:
    fh.write("#IID\n" + "\n".join(sorted(panel_ids)) + "\n")
print(f"wrote {UNREL_KEEP}  ({len(unrelated):,})")
print(f"wrote {KEEP_ALL}  ({len(panel_ids):,})")

## How related is this sample, and to what degree

Worth reading before the fits. If almost nobody is related, relatedness is not
what is producing the strange loadings and the rest of this notebook comes back
null. In that case the remaining suspects are **not** long-range LD or the MHC
(01c already excluded those) but: a batch/assay effect, residual fine-scale
population structure, or `--rm-dup`-surviving duplicated sites.

In [ ]:
bands = [(0.354, np.inf, "dup/MZ"), (0.177, 0.354, "1st"),
         (0.0884, 0.177, "2nd"), (0.0442, 0.0884, "3rd")]
rows = []
for lo, hi, name in bands:
    m = (rel_all[k_col] > lo) & (rel_all[k_col] <= hi)
    rows.append({"degree": name, "pairs": int(m.sum()),
                 "individuals": len(set(rel_all.loc[m, i_col])
                                    | set(rel_all.loc[m, j_col]))})
print(pd.DataFrame(rows).to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(rel_all[k_col].clip(0, 0.6), bins=120, log=True)
for lo, _, name in bands:
    ax.axvline(lo, color="0.6", ls="--", lw=1)
    ax.text(lo, ax.get_ylim()[1], name, rotation=90, va="top", fontsize=8)
ax.set_xlabel("KING kinship"); ax.set_ylabel("pairs (log)")
ax.set_title(f"{SAMPLE_SET} rare panel — within-sample relatedness")
plt.tight_layout(); plt.savefig(f"{OUT}/kinship_hist.png", dpi=150); plt.show()

## Refit the PCA — same variants, two sample sets

Per arm, two fits over the arm's existing `.prune.in`: `all` (reproduces the
original run) and `unrel`. `--freq counts` and `allele-wts` on both, so loadings
are directly comparable.

Runs locally. Re-runnable: each fit skips if its `.eigenvec` already exists.

In [ ]:
for arm in ARMS:
    prune_in = f"{RARE}/{arm}.prune.in"
    if not os.path.isfile(prune_in):
        print(f"skip {arm}: no {prune_in}")
        continue
    for fit, keep in (("all", KEEP_ALL), ("unrel", UNREL_KEEP)):
        out = f"{OUT}/{arm}_{fit}"
        if os.path.isfile(f"{out}.eigenvec"):
            print(f"exists, skipping: {os.path.basename(out)}.eigenvec")
            continue
        print(f"=== {arm} / {fit}")
        subprocess.run(["bash", "-c", f"""
set -eo pipefail
export PATH="$HOME/bin:$PATH"
plink2 --pfile "{PANEL}" --nonfounders \
  --keep "{keep}" --extract "{prune_in}" \
  --freq counts --pca approx {N_PCS_FIT} allele-wts \
  --threads $(nproc) --out "{out}"
echo "{arm}/{fit}: $(($(wc -l < "{out}.eigenvec") - 1)) samples scored"
"""], check=True)

## Load

In [ ]:
PC_COLS = [f"PC{k}" for k in range(1, N_PCS_FIT + 1)]

def load_fit(arm, fit):
    stem = f"{OUT}/{arm}_{fit}"
    if not os.path.isfile(f"{stem}.eigenvec"):
        return None, None
    d = pd.read_csv(f"{stem}.eigenvec", sep=r"\s+")
    idc = "#IID" if "#IID" in d.columns else "IID"
    d = d.rename(columns={idc: "person_id"})
    d["person_id"] = d["person_id"].astype(str)
    ev = np.loadtxt(f"{stem}.eigenval")
    return d[["person_id"] + PC_COLS], ev / ev.sum() * 100

pcs, pct = {}, {}
for arm in ARMS:
    for fit in ("all", "unrel"):
        p, v = load_fit(arm, fit)
        if p is not None:
            pcs[(arm, fit)], pct[(arm, fit)] = p, v
            print(f"{arm:16s} {fit:6s} n={len(p):>7,}  "
                  + "  ".join(f"PC{k+1} {v[k]:.2f}%" for k in range(5)))

DROPPED = set(dropped)
print(f"\nrelateds dropped from every 'unrel' fit: {len(DROPPED):,}")

## Loadings — the actual question

|weight| against variant index, so a loading spike shows as a vertical stripe:
one PC carried by variants in a narrow region rather than by genome-wide
structure. If `all` spikes and `unrel` does not, relatedness was the cause.

In [ ]:
def load_weights(arm, fit):
    p = f"{OUT}/{arm}_{fit}.eigenvec.allele"
    if not os.path.isfile(p):
        return None
    L = pd.read_csv(p, sep=r"\s+")
    idc = "#ID" if "#ID" in L.columns else "ID"
    parts = L[idc].str.split(":", n=2, expand=True)
    L["CHROM"] = parts[0]
    L["POS"] = pd.to_numeric(parts[1], errors="coerce")
    return L.dropna(subset=["POS"])

W = {k: load_weights(*k) for k in pcs}
W = {k: v for k, v in W.items() if v is not None}

n_show = 5
for arm in ARMS:
    keys = [k for k in W if k[0] == arm]
    if not keys:
        continue
    fig, axes = plt.subplots(n_show, len(keys), figsize=(6 * len(keys), 2.2 * n_show),
                             squeeze=False, sharex="col")
    for col, k in enumerate(keys):
        for row in range(n_show):
            ax = axes[row, col]
            ax.scatter(range(len(W[k])), W[k][f"PC{row+1}"].abs(),
                       s=0.3, alpha=0.4, rasterized=True)
            ax.set_ylabel(f"PC{row+1}")
            if row == 0:
                ax.set_title(f"{k[1]}  (n={len(pcs[k]):,})")
    plt.suptitle(f"{arm} — rare PCA loadings, |weight| by variant index")
    plt.tight_layout(); plt.savefig(f"{OUT}/loadings_{arm}.png", dpi=150); plt.show()

In [ ]:
# Concentration: share of sum(w^2) carried by the top 1% of variants.
# Diffuse genome-wide structure sits near 0.01 (uniform); a spike pushes it up.
rows = []
for (arm, fit), L in W.items():
    for k in range(1, N_PCS_FIT + 1):
        w2 = L[f"PC{k}"].to_numpy() ** 2
        n1 = max(1, len(w2) // 100)
        rows.append({"arm": arm, "fit": fit, "PC": k,
                     "top1pct_share": np.sort(w2)[-n1:].sum() / w2.sum()})
conc = (pd.DataFrame(rows)
        .pivot(index=["arm", "PC"], columns="fit", values="top1pct_share"))
print("share of squared loading mass in the top 1% of variants (uniform = 0.01):")
print(conc.round(3).to_string())

In [ ]:
# Where does the spike sit? Top-loading variants per PC in the 'all' fit.
for arm in ARMS:
    k = (arm, "all")
    if k not in W:
        continue
    print(f"\n── {arm}: most extreme loading per PC ('all' fit)")
    L = W[k]
    for pc in range(1, min(6, N_PCS_FIT + 1)):
        top = L.reindex(L[f"PC{pc}"].abs().sort_values(ascending=False).index).head(5)
        locs = ", ".join(f"{c}:{int(p/1e6)}Mb" for c, p in zip(top["CHROM"], top["POS"]))
        print(f"  PC{pc}: {locs}")

## Are the odd PCs driven by the relatives specifically?

Direct test, independent of the loadings. In the `all` fit, how much of each PC's
variance sits in the dropped individuals? Relatives are a small fraction of the
sample, so under genome-wide structure they carry their population share and no
more. Far above that share means a family-cluster PC.

In [ ]:
fam_tabs = {}
for arm in ARMS:
    if (arm, "all") not in pcs:
        continue
    d = pcs[(arm, "all")].copy()
    d["is_rel"] = d["person_id"].isin(DROPPED)
    frac = d["is_rel"].mean()
    rows = []
    for k in range(1, N_PCS_FIT + 1):
        v = d[f"PC{k}"].to_numpy(); v = v - v.mean()
        ss = (v ** 2).sum()
        top20 = d.iloc[np.argsort(-np.abs(v))[:20]]
        rows.append({"PC": k, "pct_var": pct[(arm, "all")][k - 1],
                     "var_share_in_relateds": (v[d["is_rel"].to_numpy()] ** 2).sum() / ss,
                     "expected_if_diffuse": frac,
                     "relateds_in_top20": int(top20["is_rel"].sum())})
    fam_tabs[arm] = pd.DataFrame(rows).set_index("PC")
    print(f"\n── {arm}  (relateds: {d['is_rel'].sum():,}, {100*frac:.2f}%)")
    print(fam_tabs[arm].round(4).to_string())
    flag = list(fam_tabs[arm].index[fam_tabs[arm]["var_share_in_relateds"] > 5 * frac])
    print(f"  PCs where relateds carry >5x their share: {flag}")

In [ ]:
fig, axes = plt.subplots(1, len(fam_tabs) + 1, figsize=(6 * (len(fam_tabs) + 1), 4.5),
                         squeeze=False)
for ax, (arm, t) in zip(axes[0], fam_tabs.items()):
    ax.bar(t.index, t["var_share_in_relateds"])
    ax.axhline(t["expected_if_diffuse"].iloc[0], color="crimson", ls="--", lw=1,
               label="expected if diffuse")
    ax.set_xlabel("PC"); ax.set_ylabel("share of PC variance in relateds")
    ax.set_xticks(range(1, N_PCS_FIT + 1)); ax.legend(fontsize=8); ax.set_title(arm)

ax = axes[0][-1]
for (arm, fit), v in pct.items():
    ax.plot(range(1, N_PCS_FIT + 1), v[:N_PCS_FIT], "-o", ms=4, label=f"{arm}/{fit}")
ax.set_xlabel("PC"); ax.set_ylabel("% variance explained")
ax.set_xticks(range(1, N_PCS_FIT + 1)); ax.legend(fontsize=7); ax.set_title("scree")
plt.tight_layout(); plt.savefig(f"{OUT}/relatedness_diagnostic.png", dpi=150); plt.show()

## PC pairs, relatives in red

In [ ]:
rng = np.random.default_rng(42)
for arm in ARMS:
    if (arm, "all") not in pcs:
        continue
    d = pcs[(arm, "all")].copy()
    d["is_rel"] = d["person_id"].isin(DROPPED)
    un = d[~d["is_rel"]]
    sub = un.iloc[rng.choice(len(un), size=min(40_000, len(un)), replace=False)]
    r = d[d["is_rel"]]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
    for ax, (i, j) in zip(axes, [(1, 2), (3, 4), (5, 6)]):
        ax.scatter(sub[f"PC{i}"], sub[f"PC{j}"], s=1, alpha=0.1, color="0.75",
                   rasterized=True, label="unrelated")
        ax.scatter(r[f"PC{i}"], r[f"PC{j}"], s=3, alpha=0.5, color="crimson",
                   rasterized=True, label="related (dropped)")
        ax.set_xlabel(f"PC{i}"); ax.set_ylabel(f"PC{j}")
    axes[0].legend(fontsize=8, markerscale=4)
    plt.suptitle(f"{arm} — 'all' fit, relateds in red")
    plt.tight_layout(); plt.savefig(f"{OUT}/pc_pairs_{arm}.png", dpi=150); plt.show()

## Do the two fits recover the same axes?

Correlated over the individuals the fits share (the unrelated set is a subset of
all). PCs are identified only up to sign and up to rotation within
near-degenerate subspaces, so report `max |r|` against all PCs of the other fit,
not just the diagonal.

In [ ]:
agree = {}
for arm in ARMS:
    if (arm, "all") not in pcs or (arm, "unrel") not in pcs:
        continue
    a = pcs[(arm, "all")].set_index("person_id")[PC_COLS]
    b = pcs[(arm, "unrel")].set_index("person_id")[PC_COLS]
    ids = a.index.intersection(b.index)
    R = np.corrcoef(a.loc[ids].to_numpy().T,
                    b.loc[ids].to_numpy().T)[:N_PCS_FIT, N_PCS_FIT:]
    agree[arm] = pd.DataFrame({
        "PC": range(1, N_PCS_FIT + 1),
        "diag_abs_r": np.abs(np.diag(R)),
        "max_abs_r": np.abs(R).max(axis=1),
        "best_match": np.abs(R).argmax(axis=1) + 1,
    }).set_index("PC")
    print(f"\n── {arm}: all vs unrel over {len(ids):,} shared individuals")
    print(agree[arm].round(3).to_string())

## Summary and upload

In [ ]:
parts = []
for arm in fam_tabs:
    t = fam_tabs[arm].copy()
    t["arm"] = arm
    if arm in agree:
        t["all_vs_unrel_max_r"] = agree[arm]["max_abs_r"]
    if arm in conc.index.get_level_values("arm"):
        c = conc.xs(arm, level="arm")
        for col in c.columns:
            t[f"top1pct_{col}"] = c[col]
    parts.append(t.reset_index())
summary = pd.concat(parts, ignore_index=True)

SUMMARY = f"{OUT}/rare_pca_unrelated_summary.tsv"
summary.to_csv(SUMMARY, sep="\t", index=False, float_format="%.5f")
print(summary.round(4).to_string(index=False))

subprocess.run(["bash", "-c", f"""
set -eo pipefail
gcloud storage cp "{SUMMARY}" "{OUT_GS}/rare_pca_unrelated_summary.tsv"
gcloud storage cp "{UNREL_KEEP}" "{OUT_GS}/unrelated_keep_ids.txt"
gcloud storage cp "{OUT}/"*.png "{OUT_GS}/figures/"
gcloud storage cp "{OUT}/"*.eigenvec "{OUT}/"*.eigenval "{OUT_GS}/"
"""], check=True)
print(f"\nuploaded to {OUT_GS}")

## Copy notebook to bucket

In [ ]:
_nb = os.path.expanduser('~/repos/AOU-covariance/notebooks/extra/rare_pca_unrelated.ipynb')
_gs = f'{OUT_GS}/notebooks/rare_pca_unrelated.ipynb'
subprocess.run(['gcloud', 'storage', 'cp', _nb, _gs], check=True)
print(f'notebook -> {_gs}')